# Structured Output
Modles can be requested to provide theier response in a format mathcing in a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing.
Langchain supports multiple schema type and methods for enforcing structured output.

## Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")

In [6]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

In [7]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

RunnableBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000251F5E21790>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000251F5EF1DD0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies rating out o

In [4]:
model.invoke("Provide me the details about the movie inception")

AIMessage(content='<think>\nOkay, so I need to provide details about the movie Inception. Let me start by recalling what I know about it. Inception is a 2010 movie directed by Christopher Nolan. The main actor is Leonardo DiCaprio, right? He plays Dom Cobb, a thief who steals information by entering people\'s dreams. The title, Inception, refers to the act of planting an idea into someone\'s subconscious, which is the opposite of what he usually does.\n\nThe plot involves a team that enters multiple dream layers to perform this heist. There\'s a lot of action, special effects, and some mind-bending concepts. I remember there\'s a rotating set used to show the spinning hallway fight scene. The music is by Hans Zimmer, which adds to the intensity of the movie.\n\nThere are several supporting characters. Joseph Gordon-Levitt plays Arthur, Cobb\'s right-hand man. Ellen Page is Ariadne, the architect of the dream. Tom Hardy is Eames, who can change his appearance. The story also deals with 

In [8]:
model_with_structure.invoke("Provide me the details about the movie inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

## Message output alongside parsed strcture

In [9]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """ A movie with details."""
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide details about the movie inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie "Inception." Let me check the tools available. There\'s a Movie function that requires title, year, director, and rating. I need to fill those parameters. I remember "Inception" was directed by Christopher Nolan. It came out in 2010. The rating is probably around 8.8 on IMDb. Let me confirm the director and release year to be sure. Yep, Nolan directed it, and it\'s from 2010. The rating is 8.8. So I\'ll structure the tool call with these details.\n', 'tool_calls': [{'id': 'dz2p21m6n', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 174, 'prompt_tokens': 229, 'total_tokens': 403, 'completion_time': 0.326741285, 'completion_tokens_details': {'reasoning_tokens': 126}, 'prompt_time': 0.009113296, 'prompt_tokens_det

## Nested structure

In [11]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    """ A movie with details."""
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")


model_with_structure = model.with_structured_output(MovieDetails, include_raw=True)

response = model_with_structure.invoke("Provide details about the movie inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie "Inception". I need to use the MovieDetails function to get the information. Let me check the parameters required: title, year, cast, genres, and budget. I know the title is "Inception". The year it was released is 2010. The main cast includes Leonardo DiCaprio as Dom Cobb, Joseph Gordon-Levitt as Arthur, and Ellen Page as Mary. The genres are Science Fiction and Action. The budget was around $160 million. I should structure this information into the function parameters. Let me make sure all required fields are included. Yep, title, year, cast, and genres are all there. Budget is optional, but I have that info too. Alright, time to format it into the tool call.\n', 'tool_calls': [{'id': 'nzkz1r3vx', 'function': {'arguments': '{"budget":160,"cast":[{"name":"Leonardo DiCaprio","role":"Dom Cobb"},{"name":"Joseph Gordon-Levitt","role":"Arthur"},{"name":"Ellen Page","r

## TypeDict
TypeDict provides a simpleer alternative using Python's build in typing, ideal when you don't need runtime validation

In [14]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_with_type_dict_structure = model.with_structured_output(MovieDict)

response = model_with_type_dict_structure.invoke("Please provide the details fo the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

In [15]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    """ A movie with details."""
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails, include_raw=True)

response = model_with_structure.invoke("Please provide the details fo the movie avengers")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie "Avengers." Let me see which tool I can use here. The provided tool is called MovieDetails, and it requires parameters like title, year, cast, genres, and budget.\n\nFirst, I need to figure out which version of Avengers they\'re referring to. There are several, but the most likely one is the 2012 film, also known as "The Avengers," directed by Joss Whedon. Let me confirm the title and year. The original release year was 2012. The title might just be "Avengers," but sometimes it\'s listed as "The Avengers" in some databases. I should check both possibilities, but since the user wrote "avengers" without "The," maybe it\'s the 2012 version.\n\nNext, the cast. The main cast includes Robert Downey Jr. as Iron Man, Chris Evans as Captain America, Mark Ruffalo as Hulk, Chris Hemsworth as Thor, Scarlett Johansson as Black Widow, and Jeremy Renner as Hawkeye. I need to lis

In [16]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

## DataClasses
A data class is a class typically containing mainly data, although there arent really any restrictions. You can create it uing the @dataclass decorator

In [19]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """ Contact infromation for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")


agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
    }]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='222b5769-c20a-4d27-a2cf-378b27291305'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants me to extract contact information from the given string: "John Doe, john@example.com, (555) 123-4567". Let me break this down.\n\nFirst, I need to identify the different parts. The name is John Doe. That\'s straightforward. Next, the email address is john@example.com. The phone number is (555) 123-4567. I should check if the phone number is in the correct format. The function parameters require name, email, and phone. All three are present here. The phone number has parentheses and a space, but the function doesn\'t specify formatting, so I\'ll include it as is. Everything seems to fit the required parameters. I\'ll structure the JSON with these details.\n', 'tool_calls': [{'id': 'jd7x659y0', 'function': {'ar

In [20]:
print(result['structured_response'])

name='John Doe' email='john@example.com' phone='(555) 123-4567'
